In [28]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [29]:
eng_sentences = [
    "hello",
    "thank you",
    "good morning"
]

tam_sentences = [
    "வணக்கம்",
    "நன்றி",
    "காலை வணக்கம்"
]

# add start and end tokens
tam_sentences = ["<start> " + s + " <end>" for s in tam_sentences]


In [30]:
eng_tokenizer = Tokenizer()
tam_tokenizer = Tokenizer(filters="")

eng_tokenizer.fit_on_texts(eng_sentences)
tam_tokenizer.fit_on_texts(tam_sentences)

eng_seq = eng_tokenizer.texts_to_sequences(eng_sentences)
tam_seq = tam_tokenizer.texts_to_sequences(tam_sentences)

max_eng_len = max(len(s) for s in eng_seq)
max_tam_len = max(len(s) for s in tam_seq)

encoder_input = pad_sequences(eng_seq, maxlen=max_eng_len, padding="post")
decoder_input = pad_sequences(tam_seq, maxlen=max_tam_len, padding="post")


In [31]:
latent_dim = 128

encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(len(eng_tokenizer.word_index)+1, latent_dim)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]


In [32]:
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(len(tam_tokenizer.word_index)+1, latent_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(len(tam_tokenizer.word_index)+1, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)


In [33]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

decoder_target = np.zeros(decoder_input.shape)
decoder_target[:, :-1] = decoder_input[:, 1:]
decoder_target = np.expand_dims(decoder_target, -1)

model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    epochs=300,
    batch_size=1,
    verbose=0
)


In [34]:
encoder_model = Model(encoder_inputs, encoder_states)


In [35]:
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))

decoder_outputs, h, c = decoder_lstm(
    dec_emb,
    initial_state=[decoder_state_input_h, decoder_state_input_c]
)

decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs, h, c]
)


In [36]:
index_to_tam = {v: k for k, v in tam_tokenizer.word_index.items()}

def translate(sentence):
    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_eng_len, padding="post")

    h, c = encoder_model.predict(seq, verbose=0)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = tam_tokenizer.word_index["<start>"]

    result = []

    for _ in range(max_tam_len):
        output, h, c = decoder_model.predict(
            [target_seq, h, c], verbose=0
        )

        word_index = np.argmax(output[0, 0])
        word = index_to_tam.get(word_index, "")

        if word == "<end>":
            break

        result.append(word)
        target_seq[0, 0] = word_index

    return " ".join(result)


In [37]:
print("English : hello")
print("Tamil   :", translate("hello"))

print("English : thank you")
print("Tamil   :", translate("thank you"))


English : hello
Tamil   : வணக்கம்
English : thank you
Tamil   : நன்றி
